In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import os
PROJECT_DIR = "/content/drive/MyDrive/DL_Group_MGI1"
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
print("Working in:", os.getcwd())


Working in: /content/drive/MyDrive/DL_Group_MGI1


In [ ]:
# Install dependencies if not already installed
!pip install -q torch torchvision lightning matplotlib seaborn pathlib scikit-learn scikit-image


FileNotFoundError: [Errno 2] No such file or directory: '/usr/local/lib/python3.*/site-packages/'

In [10]:
# Run this everytime you update something in the repo!
import os

REPO = "https://github.com/gabrielcastrob/Deep_learning_WUR"

# if project directory is empty clone the repo, otherwise pull the latest changes
if not os.listdir(PROJECT_DIR):
    !git clone {REPO} {PROJECT_DIR}
else:
    !git -C {PROJECT_DIR} pull

%cd {PROJECT_DIR}

print("Working in:", os.getcwd())





Already up to date.
/content
Working in: /content/drive/MyDrive/DL_Group_MGI1


Download dataset

In [11]:

"Download the UCMerced Land Use dataset if not already present. "
"The dataset will be saved in the 'ucmdata' directory. "

import os
import zipfile
import subprocess
import shutil
if not os.path.exists('ucmdata'):
    subprocess.run(['git', 'clone', 'https://git.wur.nl/lobry001/ucmdata.git'])
    os.chdir('ucmdata')

    with zipfile.ZipFile('UCMerced_LandUse.zip', 'r') as zip_ref:
        zip_ref.extractall('UCMImages')

    shutil.move('UCMImages/UCMerced_LandUse/Images', '.')
    shutil.rmtree('UCMImages')
    os.remove('README.md')
    os.remove('UCMerced_LandUse.zip')
    print(os.listdir('.'))
    UCM_images_path = "Images/"
    Multilabels_path = "LandUse_Multilabeled.txt"

Load Packages

In [12]:
 import random, shutil, zipfile
from pathlib import Path
import glob

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as tvm
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt

import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger
import torchmetrics
import re 

L.seed_everything(42, workers=True)
DEVICE = "gpu" if torch.cuda.is_available() else "cpu"
print("Accelerator:", DEVICE)
if DEVICE == "gpu":
    print("GPU:", torch.cuda.get_device_name(0))

INFO: Seed set to 42
INFO:lightning.fabric.utilities.seed:Seed set to 42


Accelerator: gpu
GPU: Tesla T4


Class list and dataset

In [13]:
class UCMMultilabelDataset(Dataset):
    """
    UCM multilabel land-use dataset.
 
    Folder layout:
        ucmdata/
            Images/
                agricultural/   agricultural00.tif … agricultural99.tif
                airplane/       airplane00.tif      … airplane99.tif
                …               (21 subfolders, already in correct order)
            LandUse_Multilabeled.txt
 
    Label file format (tab-separated):
        IMAGE\LABEL   airplane   bare-soil   buildings   …   water
        agricultural00    0           0            0      …     0
        …
 
    Strategy
    --------
    1. Parse the label file → class names from the header (cols 1…end,
       skipping "IMAGE\LABEL") + label matrix (N × C).
    2. Walk ucmdata/Images/ subfolder-by-subfolder in sorted order, collecting
       every image path into a flat list — same traversal order as the txt file.
    3. Pair image_paths[i]  ↔  label_matrix[i]  by position (no name matching).
    """
 
    def __init__(
        self,
        root_dir: str = "ucmdata",
        label_file: str = "LandUse_Multilabeled.txt",
        transform=None,
        image_ext: str = ".tif",
    ):
        self.root_dir   = root_dir
        self.images_dir = os.path.join(root_dir, "Images")
        self.transform  = transform
        self.image_ext  = image_ext
 
        # ── 1. Parse label file ──────────────────────────────────────────
        label_path = os.path.join(root_dir, label_file)
        self.class_names, self.label_matrix = self._parse_labels(label_path)
        self.num_classes = len(self.class_names)
 
        # ── 2. Collect image paths in sorted subfolder order ─────────────
        self.image_paths = self._collect_image_paths()
 
        # ── 3. Sanity check ──────────────────────────────────────────────
        assert len(self.image_paths) == len(self.label_matrix), (
            f"Mismatch: {len(self.image_paths)} images found but "
            f"{len(self.label_matrix)} label rows in the txt file."
        )
 
    # ------------------------------------------------------------------ #
    def _parse_labels(self, label_path: str):
        """
        Returns:
            class_names  – list[str], length C  (column headers, cols 1…end)
            label_matrix – torch.FloatTensor, shape (N, C)
        """
        with open(label_path, "r") as f:
            lines = [l.strip() for l in f if l.strip()]
 
        # Header row → class names (skip the first column "IMAGE\LABEL")
        header      = lines[0].split("\t")
        class_names = header[1:]          # ['airplane', 'bare-soil', …, 'water']
 
        # Data rows → label matrix (parts[0] is the image name, ignored here)
        rows = []
        for line in lines[1:]:
            parts = line.split("\t")
            label_vals = list(map(int, parts[1:]))
            rows.append(label_vals)
 
        label_matrix = torch.tensor(rows, dtype=torch.float32)  # (N, C)
        return class_names, label_matrix
 
    # ------------------------------------------------------------------ #
    def _collect_image_paths(self) -> list:
        """
        Walks ucmdata/Images/ subfolder-by-subfolder in sorted order.
        Within each subfolder images are also sorted — matching the txt order.
        Returns a flat list of absolute image paths.
        """
        image_paths = []
 
        subfolders = sorted(
            entry.name
            for entry in os.scandir(self.images_dir)
            if entry.is_dir()
        )
 
        for subfolder in subfolders:
            folder_path = os.path.join(self.images_dir, subfolder)
            files = sorted(
                fname
                for fname in os.listdir(folder_path)
                if fname.lower().endswith(self.image_ext)
            )
            for fname in files:
                image_paths.append(os.path.join(folder_path, fname))
 
        return image_paths
 
    # ------------------------------------------------------------------ #
    def __len__(self) -> int:
        return len(self.image_paths)
 
    def __getitem__(self, idx: int):
        img_path = self.image_paths[idx]
        labels   = self.label_matrix[idx]          # float32 tensor, shape (C,)
 
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
 
        return image, labels
 
    # ------------------------------------------------------------------ #
    # Utility helpers
    # ------------------------------------------------------------------ #
    def get_class_names(self) -> list:
        return self.class_names
 
    def get_class_weights(self) -> torch.Tensor:
        """
        Inverse-frequency pos_weight per class for BCEWithLogitsLoss.
        Shape: (num_classes,)
        """
        pos = self.label_matrix.sum(dim=0).clamp(min=1)
        neg = (len(self) - self.label_matrix.sum(dim=0)).clamp(min=1)
        return neg / pos
 
    def verify_alignment(self, n_samples: int = 5):
        """
        Prints the first n_samples rows so you can visually cross-check
        image names against the txt file.
        """
        print(f"{'Image path':<55}  {'Active labels'}")
        print("-" * 80)
        for i in range(min(n_samples, len(self))):
            active = [self.class_names[j]
                      for j, v in enumerate(self.label_matrix[i]) if v == 1]
            short = os.path.join(*self.image_paths[i].split(os.sep)[-2:])
            print(f"{short:<55}  {active}")
 

<>:14: SyntaxWarning: invalid escape sequence '\L'
<>:14: SyntaxWarning: invalid escape sequence '\L'
/tmp/ipykernel_19509/1943201494.py:14: SyntaxWarning: invalid escape sequence '\L'
  IMAGE\LABEL   airplane   bare-soil   buildings   …   water


In [15]:
basedataset = UCMMultilabelDataset()
print("Number of samples:", len(basedataset))
print("Class names:", basedataset.get_class_names())
print("Class weights:", basedataset.get_class_weights())
basedataset.verify_alignment()



Number of samples: 2100
Class names: ['airplane', 'bare-soil', 'buildings', 'cars', 'chaparral', 'court', 'dock', 'field', 'grass', 'mobile-home', 'pavement', 'sand', 'sea', 'ship', 'tanks', 'trees', 'water']
Class weights: tensor([20.0000,  1.9248,  2.0391,  1.3702, 17.2609, 19.0000, 20.0000, 19.3883,
         1.1538, 19.5882,  0.6154,  6.1429, 20.0000, 19.5882, 20.0000,  1.0813,
         9.3448])
Image path                                               Active labels
--------------------------------------------------------------------------------
agricultural/agricultural00.tif                          ['field', 'trees']
agricultural/agricultural01.tif                          ['field']
agricultural/agricultural02.tif                          ['field']
agricultural/agricultural03.tif                          ['field']
agricultural/agricultural04.tif                          ['trees']
